In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, classification_report, 
                              confusion_matrix, roc_auc_score, roc_curve)
from imblearn.over_sampling import SMOTE
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Limpeza rápida (mesmo que fizemos no notebook 1)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df = df.drop('customerID', axis=1)

print(f"Dataset: {df.shape[0]} linhas x {df.shape[1]} colunas")

Dataset: 7043 linhas x 20 colunas


## Machine Learning — Previsão de Churn

**Objetivo:** Prever se um cliente vai cancelar ANTES de ele cancelar, 
permitindo que a empresa tome ações preventivas de retenção.

1. Vamos lidar com dados desbalanceados usando SMOTE
2. Vamos testar 3 modelos e compará-los com validação cruzada
3. Vamos usar GridSearchCV para otimização automática de hiperparâmetros
4. Vamos gerar curvas ROC para comparação visual dos modelos

In [2]:
# Etapa 1: Preparar os dados para ML

# Converter Churn para 0/1
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Separar colunas categóricas e numéricas
categoricas = df.select_dtypes(include='object').columns.tolist()
numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']

print(f"Categóricas ({len(categoricas)}): {categoricas}")
print(f"Numéricas ({len(numericas)}): {numericas}")
print(f"\nChurn convertido: {df['Churn'].value_counts().to_dict()}")

Categóricas (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Numéricas (3): ['tenure', 'MonthlyCharges', 'TotalCharges']

Churn convertido: {0: 5174, 1: 1869}


In [3]:
# Etapa 2: Encoding das variáveis categóricas

# Remover gender e PhoneService (não são significativos, como vimos na análise)
df = df.drop(['gender', 'PhoneService'], axis=1)
categoricas.remove('gender')
categoricas.remove('PhoneService')

# One-Hot Encoding
df_ml = pd.get_dummies(df, columns=categoricas, drop_first=True)

print(f"Features criadas: {df_ml.shape[1] - 1}")
print(f"Linhas: {df_ml.shape[0]}")

Features criadas: 28
Linhas: 7043


### Etapa 3: Divisão dos dados e balanceamento com SMOTE

**O problema:** 73.5% dos dados são "não cancelou" e 26.5% "cancelou". 
Se treinarmos assim, o modelo aprende a chutar "não cancela" e já acerta 73%.

**SMOTE** (Synthetic Minority Oversampling Technique) resolve isso criando 
exemplos sintéticos da classe minoritária (churn). Ele olha pra clientes 
que cancelaram, encontra vizinhos parecidos, e cria novos registros 
"intermediários" entre eles. Assim o modelo aprende igualmente os dois padrões.

In [4]:
# Separar features e alvo
X = df_ml.drop('Churn', axis=1)
y = df_ml['Churn']

# Dividir em treino (80%) e teste (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Treino: {X_train.shape[0]} | Teste: {X_test.shape[0]}")
print(f"\nAntes do SMOTE (treino):")
print(f"  Não cancelou: {(y_train == 0).sum()}")
print(f"  Cancelou:     {(y_train == 1).sum()}")

# Aplicar SMOTE apenas no treino
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"\nDepois do SMOTE (treino):")
print(f"  Não cancelou: {(y_train_bal == 0).sum()}")
print(f"  Cancelou:     {(y_train_bal == 1).sum()}")

Treino: 5634 | Teste: 1409

Antes do SMOTE (treino):
  Não cancelou: 4139
  Cancelou:     1495

Depois do SMOTE (treino):
  Não cancelou: 4139
  Cancelou:     4139


In [5]:
# Etapa 4: Normalizar os dados
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Etapa 5: Treinar 3 modelos e comparar
modelos = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42)
}

resultados = {}

for nome, modelo in modelos.items():
    # Treinar
    modelo.fit(X_train_scaled, y_train_bal)
    
    # Prever
    y_pred = modelo.predict(X_test_scaled)
    y_proba = modelo.predict_proba(X_test_scaled)[:, 1]
    
    # Métricas
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    # Validação cruzada (5 folds)
    cv_scores = cross_val_score(modelo, X_train_scaled, y_train_bal, cv=5, scoring='roc_auc')
    
    resultados[nome] = {
        'accuracy': acc,
        'auc': auc,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'y_pred': y_pred,
        'y_proba': y_proba
    }
    
    print(f"\n{'='*50}")
    print(f"  {nome}")
    print(f"{'='*50}")
    print(f"  Acurácia:        {acc*100:.1f}%")
    print(f"  AUC-ROC:         {auc:.3f}")
    print(f"  CV AUC (5-fold): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")


  Logistic Regression
  Acurácia:        74.8%
  AUC-ROC:         0.808
  CV AUC (5-fold): 0.899 ± 0.056

  Random Forest
  Acurácia:        76.7%
  AUC-ROC:         0.836
  CV AUC (5-fold): 0.919 ± 0.040

  Gradient Boosting
  Acurácia:        77.6%
  AUC-ROC:         0.817
  CV AUC (5-fold): 0.927 ± 0.055


### Etapa 6: Otimização com GridSearchCV
GridSearchCV testa automaticamente várias combinações de parâmetros 
e encontra a melhor. É como tentar todas as combinações de tempero 
numa receita e ficar com a mais saborosa.

In [6]:
# GridSearchCV no Gradient Boosting
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'min_samples_split': [5, 10]
}

grid_search = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train_bal)

print(f"\nMelhores parâmetros: {grid_search.best_params_}")
print(f"Melhor AUC-ROC (CV): {grid_search.best_score_:.3f}")

# Avaliar no teste
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test_scaled)
y_proba_best = best_model.predict_proba(X_test_scaled)[:, 1]

print(f"\nPerformance no teste:")
print(f"  Acurácia: {accuracy_score(y_test, y_pred_best)*100:.1f}%")
print(f"  AUC-ROC:  {roc_auc_score(y_test, y_proba_best):.3f}")
print(f"\n{classification_report(y_test, y_pred_best, target_names=['Fica', 'Cancela'])}")

Fitting 5 folds for each of 54 candidates, totalling 270 fits

Melhores parâmetros: {'learning_rate': 0.05, 'max_depth': 7, 'min_samples_split': 5, 'n_estimators': 300}
Melhor AUC-ROC (CV): 0.930

Performance no teste:
  Acurácia: 77.7%
  AUC-ROC:  0.819

              precision    recall  f1-score   support

        Fica       0.85      0.84      0.85      1035
     Cancela       0.58      0.60      0.59       374

    accuracy                           0.78      1409
   macro avg       0.72      0.72      0.72      1409
weighted avg       0.78      0.78      0.78      1409



In [7]:
# Curva ROC — comparação visual dos modelos
fig = go.Figure()

cores = {'Logistic Regression': '#3B82F6', 'Random Forest': '#10B981', 'Gradient Boosting': '#8B5CF6'}

for nome, res in resultados.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, name=f"{nome} (AUC={res['auc']:.3f})",
        line=dict(color=cores[nome], width=2.5)
    ))

# Adicionar o modelo otimizado
fpr_best, tpr_best, _ = roc_curve(y_test, y_proba_best)
fig.add_trace(go.Scatter(
    x=fpr_best, y=tpr_best, name=f"GB Otimizado (AUC={roc_auc_score(y_test, y_proba_best):.3f})",
    line=dict(color='#EF4444', width=3)
))

# Linha diagonal (modelo aleatório)
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], name='Aleatório (AUC=0.500)',
    line=dict(color='gray', width=1, dash='dash')
))

fig.update_layout(
    title=dict(text='Curva ROC — Comparação dos Modelos', font=dict(size=20)),
    template='plotly_white',
    xaxis=dict(title='Taxa de Falsos Positivos'),
    yaxis=dict(title='Taxa de Verdadeiros Positivos'),
    legend=dict(x=0.55, y=0.05),
    height=500
)

fig.show()

### Etapa 7: Explicabilidade com SHAP
SHAP (SHapley Additive exPlanations) explica a contribuição de cada variável 
na previsão do modelo. Em vez de apenas dizer "esse cliente vai cancelar", 
o SHAP diz "esse cliente vai cancelar PORQUE tem contrato mensal, 
paga por electronic check e tem pouco tempo como cliente".
Isso é essencial no mundo corporativo — gestores precisam entender o porquê.

In [8]:
import shap

# Usar o modelo otimizado
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_scaled)

# Importância média de cada feature
shap_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': np.abs(shap_values).mean(axis=0)
}).sort_values('importance', ascending=False).head(15)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=shap_importance['importance'].values[::-1],
        y=shap_importance['feature'].values[::-1],
        orientation='h',
        marker_color='#8B5CF6'
    )
)

fig.update_layout(
    title=dict(text='Top 15 Variáveis — Importância SHAP', font=dict(size=20)),
    template='plotly_white',
    xaxis=dict(title='Impacto médio no modelo (|SHAP value|)'),
    height=500
)

fig.show()

### Insights — Explicabilidade SHAP

O SHAP confirma e quantifica os achados da análise estatística:

1. **PaymentMethod_Electronic check** é o fator #1 — confirma o teste chi-quadrado 
   que mostrou 45.3% de churn nesse grupo. Possível causa: electronic check exige 
   ação manual todo mês, criando oportunidades de cancelamento
2. **InternetService_Fiber optic** em #2 — o serviço premium que gera mais cancelamento
3. **MonthlyCharges** em #3 — quanto mais caro, mais risco
4. **tenure** em #4 — tempo como cliente é fator protetor
5. **Contract_Two year** aparece — confirma que contratos longos reduzem churn

**A análise estatística e o ML contam a mesma história.** Isso valida os dois: 
a estatística identificou os padrões, o ML confirmou e quantificou a importância.

In [9]:
# Salvar o modelo e o scaler
import joblib

joblib.dump(best_model, '../models/gradient_boosting_churn.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

# Salvar as colunas do modelo (importante pra usar no app)
feature_names = X.columns.tolist()
joblib.dump(feature_names, '../models/feature_names.pkl')

print("Modelo salvo: models/gradient_boosting_churn.pkl")
print("Scaler salvo: models/scaler.pkl")
print(f"Features: {len(feature_names)} variáveis")

Modelo salvo: models/gradient_boosting_churn.pkl
Scaler salvo: models/scaler.pkl
Features: 28 variáveis
